
# 08 — Behavior and Customer Prediction

Train conventional predictive models on Notebook 07's chronological modeling tables.

For both Retailrocket and Online Retail II:

- Logistic Regression baseline
- HistGradientBoostingClassifier
- validation-selected threshold
- ROC-AUC, PR-AUC, F1, Brier score, Precision@10%
- final test evaluation


In [ ]:

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path.home() / "Desktop" / "resume_projects" / "ProductPulse"
PROCESSED_02 = PROJECT_ROOT / "data" / "processed" / "02_cleaned"
PROCESSED_07 = PROJECT_ROOT / "data" / "processed" / "07_modeling"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

PROCESSED_07.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

print("PROJECT_ROOT:", PROJECT_ROOT)


In [ ]:

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import joblib

MODELS_DIR = ARTIFACTS_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

def best_f1_threshold(y_true, prob):
    precision, recall, thresholds = precision_recall_curve(y_true, prob)
    f1 = 2 * precision[:-1] * recall[:-1] / (
        precision[:-1] + recall[:-1] + 1e-12
    )
    idx = int(np.nanargmax(f1))
    return float(thresholds[idx])

def precision_at_fraction(y_true, prob, fraction=0.10):
    n = max(1, int(np.ceil(len(prob) * fraction)))
    idx = np.argsort(prob)[::-1][:n]
    return float(np.mean(np.asarray(y_true)[idx]))

def evaluate(y, p, threshold, model, split):
    pred = (p >= threshold).astype(int)
    return {
        "model": model,
        "split": split,
        "roc_auc": roc_auc_score(y, p),
        "pr_auc": average_precision_score(y, p),
        "f1": f1_score(y, pred),
        "brier": brier_score_loss(y, p),
        "precision_at_10pct": precision_at_fraction(y, p),
        "threshold": threshold,
        "positive_rate": float(np.mean(y)),
    }

def numeric_x(df, cols):
    return df[cols].replace([np.inf, -np.inf], np.nan)

def make_lr():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        )),
    ])

def make_hgb():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingClassifier(
            learning_rate=0.08,
            max_iter=150,
            max_leaf_nodes=31,
            l2_regularization=1.0,
            random_state=42,
        )),
    ])


## Retailrocket conversion propensity

In [ ]:

rr = pd.read_parquet(PROCESSED_07 / "retailrocket_conversion_features.parquet")

rr_features = [
    "total_events", "unique_items", "view", "addtocart", "transaction",
    "views_7d", "carts_7d", "transactions_7d",
    "recency_hours", "tenure_days",
    "cart_per_view", "transaction_per_view",
]
target = "target_future_transaction"

train = rr[rr["split"] == "train"].copy()
val = rr[rr["split"] == "validation"].copy()
test = rr[rr["split"] == "test"].copy()

Xtr, ytr = numeric_x(train, rr_features), train[target].to_numpy()
Xv, yv = numeric_x(val, rr_features), val[target].to_numpy()
Xt, yt = numeric_x(test, rr_features), test[target].to_numpy()

rr_lr, rr_hgb = make_lr(), make_hgb()
rr_lr.fit(Xtr, ytr)
rr_hgb.fit(Xtr, ytr)

rr_pv_lr = rr_lr.predict_proba(Xv)[:, 1]
rr_pv_hgb = rr_hgb.predict_proba(Xv)[:, 1]
rr_th_lr = best_f1_threshold(yv, rr_pv_lr)
rr_th_hgb = best_f1_threshold(yv, rr_pv_hgb)

rr_pt_lr = rr_lr.predict_proba(Xt)[:, 1]
rr_pt_hgb = rr_hgb.predict_proba(Xt)[:, 1]

rr_metrics = pd.DataFrame([
    evaluate(yv, rr_pv_lr, rr_th_lr, "LogisticRegression", "validation"),
    evaluate(yv, rr_pv_hgb, rr_th_hgb, "HistGradientBoosting", "validation"),
    evaluate(yt, rr_pt_lr, rr_th_lr, "LogisticRegression", "test"),
    evaluate(yt, rr_pt_hgb, rr_th_hgb, "HistGradientBoosting", "test"),
])
display(rr_metrics)


In [ ]:

rr_best_name = (
    rr_metrics[rr_metrics["split"] == "validation"]
    .sort_values("pr_auc", ascending=False)
    .iloc[0]["model"]
)
if rr_best_name == "HistGradientBoosting":
    rr_best, rr_test_prob, rr_threshold = rr_hgb, rr_pt_hgb, rr_th_hgb
else:
    rr_best, rr_test_prob, rr_threshold = rr_lr, rr_pt_lr, rr_th_lr

rr_top_prospects = test[["visitorid", "cutoff", target]].copy()
rr_top_prospects["purchase_probability"] = rr_test_prob
rr_top_prospects = (
    rr_top_prospects.sort_values("purchase_probability", ascending=False)
    .head(100)
)

joblib.dump(
    {
        "model": rr_best,
        "features": rr_features,
        "threshold": rr_threshold,
        "model_name": rr_best_name,
    },
    MODELS_DIR / "retailrocket_conversion_model.joblib",
)
print("Best:", rr_best_name)
display(rr_top_prospects.head(20))


## Online Retail repeat-purchase propensity

In [ ]:

or_df = pd.read_parquet(PROCESSED_07 / "online_retail_customer_features.parquet")

or_features = [
    "orders", "revenue", "units", "unique_products", "avg_order_value",
    "recency_days", "tenure_days", "orders_30d", "revenue_30d", "units_30d",
]
target_or = "target_future_purchase"

train_or = or_df[or_df["split"] == "train"].copy()
val_or = or_df[or_df["split"] == "validation"].copy()
test_or = or_df[or_df["split"] == "test"].copy()

Xtr, ytr = numeric_x(train_or, or_features), train_or[target_or].to_numpy()
Xv, yv = numeric_x(val_or, or_features), val_or[target_or].to_numpy()
Xt, yt = numeric_x(test_or, or_features), test_or[target_or].to_numpy()

or_lr, or_hgb = make_lr(), make_hgb()
or_lr.fit(Xtr, ytr)
or_hgb.fit(Xtr, ytr)

or_pv_lr = or_lr.predict_proba(Xv)[:, 1]
or_pv_hgb = or_hgb.predict_proba(Xv)[:, 1]
or_th_lr = best_f1_threshold(yv, or_pv_lr)
or_th_hgb = best_f1_threshold(yv, or_pv_hgb)

or_pt_lr = or_lr.predict_proba(Xt)[:, 1]
or_pt_hgb = or_hgb.predict_proba(Xt)[:, 1]

or_metrics = pd.DataFrame([
    evaluate(yv, or_pv_lr, or_th_lr, "LogisticRegression", "validation"),
    evaluate(yv, or_pv_hgb, or_th_hgb, "HistGradientBoosting", "validation"),
    evaluate(yt, or_pt_lr, or_th_lr, "LogisticRegression", "test"),
    evaluate(yt, or_pt_hgb, or_th_hgb, "HistGradientBoosting", "test"),
])
display(or_metrics)


In [ ]:

or_best_name = (
    or_metrics[or_metrics["split"] == "validation"]
    .sort_values("pr_auc", ascending=False)
    .iloc[0]["model"]
)
if or_best_name == "HistGradientBoosting":
    or_best, or_test_prob, or_threshold = or_hgb, or_pt_hgb, or_th_hgb
else:
    or_best, or_test_prob, or_threshold = or_lr, or_pt_lr, or_th_lr

or_top_customers = test_or[
    ["CustomerID", "cutoff", target_or, "future_revenue"]
].copy()
or_top_customers["repeat_purchase_probability"] = or_test_prob
or_top_customers = (
    or_top_customers.sort_values("repeat_purchase_probability", ascending=False)
    .head(100)
)

joblib.dump(
    {
        "model": or_best,
        "features": or_features,
        "threshold": or_threshold,
        "model_name": or_best_name,
    },
    MODELS_DIR / "online_retail_repeat_purchase_model.joblib",
)
print("Best:", or_best_name)
display(or_top_customers.head(20))


## Comparison

In [ ]:

model_comparison = pd.concat([
    rr_metrics.assign(task="Retailrocket future transaction"),
    or_metrics.assign(task="Online Retail future purchase"),
], ignore_index=True)
display(model_comparison)

# PR curve for Retailrocket
fig_pr, ax = plt.subplots(figsize=(8, 5))
p1, r1, _ = precision_recall_curve(
    rr.loc[rr["split"] == "test", "target_future_transaction"], rr_pt_lr
)
p2, r2, _ = precision_recall_curve(
    rr.loc[rr["split"] == "test", "target_future_transaction"], rr_pt_hgb
)
ax.plot(r1, p1, label="Logistic")
ax.plot(r2, p2, label="HistGradientBoosting")
ax.set_title("Retailrocket Precision–Recall Curves")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.legend()
fig_pr.tight_layout()
plt.show()

test_plot = model_comparison[model_comparison["split"] == "test"].copy()
test_plot["label"] = test_plot["task"] + " — " + test_plot["model"]

fig_metrics, ax = plt.subplots(figsize=(9, 5))
ax.bar(np.arange(len(test_plot)), test_plot["pr_auc"])
ax.set_xticks(np.arange(len(test_plot)))
ax.set_xticklabels(test_plot["label"], rotation=25, ha="right")
ax.set_ylabel("PR-AUC")
ax.set_title("Test PR-AUC Comparison")
fig_metrics.tight_layout()
plt.show()


## Save compact Notebook 08 results

In [ ]:

RESULTS_DIR = PROJECT_ROOT / "results" / "08_behavior_and_customer_prediction"
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RESULT_TABLES = {
    "retailrocket_model_metrics": rr_metrics,
    "online_retail_model_metrics": or_metrics,
    "model_comparison": model_comparison,
    "retailrocket_top_prospects": rr_top_prospects,
    "online_retail_top_customers": or_top_customers,
}
RESULT_FIGURES = {
    "retailrocket_precision_recall": fig_pr,
    "test_pr_auc_comparison": fig_metrics,
}

for name, table in RESULT_TABLES.items():
    table.to_csv(TABLES_DIR / f"{name}.csv", index=False)
for name, fig in RESULT_FIGURES.items():
    fig.savefig(FIGURES_DIR / f"{name}.png", dpi=200, bbox_inches="tight")

print("Tables saved :", len(RESULT_TABLES))
print("Figures saved:", len(RESULT_FIGURES))
print("Results:", RESULTS_DIR)
print("Models:", MODELS_DIR)
